In [1]:
import os
from dotenv import load_dotenv
from mp_api.client import MPRester
import pandas as pd

load_dotenv()

MP_API_KEY = os.getenv("MP_API_KEY")

with MPRester(MP_API_KEY) as mpr:
    print("API Connection Sucessful")

C:\Users\danma\Documents\Dan\Projects\Band Gap AI Stack\Multimodal GNN\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


API Connection Sucessful


In [2]:
# Define the fields we want to retrieve
fields = [
    "material_id",
    "formula_pretty",
    "structure",                # GNN Input
    "density",                  # MLP Input 1
    "nelements",                # MLP Input 2
    "symmetry",                 # MLP Input 3 (to get crystal_system)
    "band_gap",                 # Target 1
    "formation_energy_per_atom" # Target 2 (for stability screening)
]

In [3]:
with MPRester(MP_API_KEY) as mpr:
    # Query for semiconductor candidates
    # We filter for 'is_stable' to ensure high-quality training data
    docs = mpr.materials.summary.search(
        band_gap=(0, 5.0),      # Target range for hardware
        is_stable=True,           # Must be manufacture-able
        fields=fields
    )

Retrieving SummaryDoc documents: 100%|████████████████████████████████████████████████████████████████████| 32613/32613 [00:47<00:00, 686.18it/s]


In [4]:
# Convert to a list of dictionaries
data_list = []
for doc in docs:
    data_list.append({
        "mp_id": str(doc.material_id),
        "formula": doc.formula_pretty,
        "structure": doc.structure,
        "density": doc.density,
        "nelements": doc.nelements,
        "crystal_system": str(doc.symmetry.crystal_system), # Extracted from symmetry
        "band_gap": doc.band_gap,
        "formation_energy": doc.formation_energy_per_atom
    })

In [5]:
df = pd.DataFrame(data_list)
df.shape

(32613, 8)

In [6]:
df.head(5)

,mp_id,formula,structure,density,nelements,crystal_system,band_gap,formation_energy
0,mp-862690,Ac,"[[0. 0. 0.] Ac, [-3.50000702e-07 2.33598723e+...",8.170182,1,Hexagonal,0.0,0.000000
1,mp-1183076,Ac2AgPb,"[[6.10084398 6.10084398 6.10084398] Ac, [2.033...",9.490476,3,Cubic,0.0,-0.480278
2,mp-1183086,Ac2CdHg,"[[6.07823546 6.07823546 6.07823546] Ac, [2.026...",9.570977,3,Cubic,0.0,-0.433682
3,mp-862786,Ac2CuGe,"[[3.8482657 3.8482657 3.8482657] Ac, [0. 0. 0....",8.598305,3,Cubic,0.0,-0.386251
4,mp-861883,Ac2CuIr,"[[3.73365347 3.73365347 3.73365347] Ac, [0. 0....",11.322190,3,Cubic,0.0,-0.342299


In [7]:
int(df.isnull().sum().sum())

0

In [8]:
df.dtypes.value_counts()

str        3
float64    3
object     1
int64      1
Name: count, dtype: int64

In [9]:
from pathlib import Path
cwd = Path.cwd()

SAVE_PATH = cwd.parent / "data" / "materials_dataset_v2.pkl"

In [10]:
df.to_pickle(SAVE_PATH)

In [11]:
df[['formula', 'crystal_system', 'band_gap']].head()

,formula,crystal_system,band_gap
0,Ac,Hexagonal,0.0
1,Ac2AgPb,Cubic,0.0
2,Ac2CdHg,Cubic,0.0
3,Ac2CuGe,Cubic,0.0
4,Ac2CuIr,Cubic,0.0
